# GraphRAG Pipeline — Team Walkthrough & Unit Tests

This notebook documents **how our production GraphRAG stack works** (upload → async ingestion → graph + vectors + communities → query routing → retrieval strategies → grounded answers) and runs **assert-based unit tests** your team can execute in Jupyter.

| Mode | What you need |
|------|----------------|
| **Unit tests (default)** | Python deps only; LLM/Neo4j/Milvus are mocked |
| **Integration (optional)** | `.env` with Neo4j, Milvus, Redis, MinIO, LLM; API on port 8002 |

**Related code:** `src/workers/ingestion_worker.py`, `src/retrieval/graph.py`, `src/retrieval/hybrid.py`, `src/services/query_service.py`, `gradio_app.py`


## Architecture (production path)

```mermaid
flowchart TB
    subgraph upload["Upload (FastAPI)"]
        F[PDF/DOCX/PPTX/XLSX] --> MINIO[MinIO original]
        F --> REDIS[Redis ingestion job]
    end

    subgraph worker["Ingestion worker"]
        DL[Download from MinIO]
        PP[Preprocess: MinerU / direct extract]
        CH[Chunk TextUnits]
        GB[LLM graph extract → Neo4j]
        MV[Embed chunks → Milvus]
        CD[Leiden communities → Neo4j]
        CR[Community summaries]
        DL --> PP --> CH --> GB
        CH --> MV
        GB --> CD --> CR
    end

    subgraph query["Query API"]
        Q[User question]
        EN[Query enhancer optional]
        RT{Router: LOCAL vs GLOBAL}
        LS[Local: entity paths + LLM]
        GS[Global: community reports + LLM]
        HY[Hybrid: graph + vector parallel]
        RR[Rerank CrossEncoder]
        GEN[Final LLM answer]
        AG[Deep agent: KB then Tavily]
        Q --> EN
        EN --> RT
        RT -->|LOCAL| LS
        RT -->|GLOBAL| GS
        EN --> HY --> RR --> GEN
        EN --> AG
    end

    REDIS --> worker
    MINIO --> worker
    GB --> query
    MV --> query
    CR --> query
```


In [ ]:
# Bootstrap: project root on sys.path
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / "src" / "services" / "query_service.py").exists():
            PROJECT_ROOT = parent
            break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_INTEGRATION = os.getenv("GRAPHRAG_RUN_INTEGRATION", "").lower() in ("1", "true", "yes")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_INTEGRATION:", RUN_INTEGRATION)


In [ ]:
# Lightweight test runner for notebook cells
class TestResults:
    def __init__(self):
        self.passed = []
        self.failed = []

    def check(self, name, condition, detail=""):
        if condition:
            self.passed.append(name)
            print(f"  PASS  {name}")
        else:
            self.failed.append((name, detail))
            print(f"  FAIL  {name} — {detail}")

    def summary(self):
        print("\n" + "=" * 60)
        print(f"Passed: {len(self.passed)} | Failed: {len(self.failed)}")
        if self.failed:
            for name, detail in self.failed:
                print(f"  - {name}: {detail}")
            raise AssertionError(f"{len(self.failed)} test(s) failed")
        print("All tests passed.")

results = TestResults()


## 1. Codebase map (what to read first)

| Layer | Path | Role |
|-------|------|------|
| API | `app/main.py`, `app/routes/upload.py`, `app/routes/query.py` | HTTP upload + `/query`, `/query/hybrid`, `/query/agent` |
| Upload | `src/services/upload_service.py` | Save → MinIO → enqueue Redis job |
| Worker | `src/workers/ingestion_worker.py` | Full index pipeline per document |
| Ingest | `src/ingestion/ingest.py` | Chunk + Milvus embed |
| Graph | `src/graph/graph_builder.py` | Per-chunk LLM → Neo4j |
| Communities | `src/graph/community.py` | Leiden + summaries for global search |
| Router | `src/retrieval/graph.py` | `classify_query` → local or global |
| Vector | `src/retrieval/vector.py` | Milvus COSINE search |
| Hybrid | `src/retrieval/hybrid.py` | Parallel graph + vector → rerank → answer |
| Agent | `src/agent/agent.py` | Deep agent + KB tool + Tavily fallback |
| UI | `gradio_app.py` | Calls API upload + `/query/agent` |

**Notebook demo (`graphrag_flow.ipynb`)** is a separate teaching path (LangChain + Meezan branch search). This repo's **production** path is the table above.


## 2. Upload & ingestion — unit tests

Upload flow (no LLM):
1. Validate extension (`.pdf`, `.docx`, `.pptx`, `.xlsx`)
2. Assign `document_id`, register in Milvus metadata layer
3. Push original to MinIO, enqueue Redis job

Worker flow:
1. Download → preprocess → chunk
2. `build_graph_from_chunks` → Neo4j
3. `embed_and_ingest` → Milvus
4. Leiden communities + summaries → Neo4j `Community` nodes


In [ ]:
# Upload contract + ingestion job shape
ALLOWED = {".pdf", ".docx", ".pptx", ".xlsx"}

def is_allowed_upload(filename: str) -> bool:
    ext = os.path.splitext(filename)[1].lower()
    return ext in ALLOWED

results.check("pdf allowed", is_allowed_upload("report.PDF"))
results.check("xlsx allowed", is_allowed_upload("branches.xlsx"))
results.check("txt rejected", not is_allowed_upload("notes.txt"))

required_job_keys = {"document_id", "filename", "source_key", "uploaded_by", "uploaded_at"}
sample_job = {
    "document_id": "uuid",
    "filename": "sample.pdf",
    "source_key": "minio/key",
    "uploaded_by": "api-upload",
    "uploaded_at": "2026-01-01T00:00:00Z",
}
results.check("job keys complete", required_job_keys <= set(sample_job.keys()))

results.summary()


In [ ]:
# Chunking unit test (same logic as src/ingestion/ingest.chunk_documents)
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = "GraphRAG combines graphs and retrieval. " * 40
docs = [Document(page_content=long_text, metadata={"page": 1})]
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_documents(docs)

results = TestResults()
results.check("produces multiple chunks", len(chunks) >= 2)
results.check("preserves metadata page", all(c.metadata.get("page") == 1 for c in chunks))
results.check("chunk size bounded", all(len(c.page_content) <= 150 for c in chunks))
results.summary()


In [ ]:
# Graph builder: empty extraction must be reported as failure
# (same rule as tests/test_graph_builder_zero_nodes.py)
from types import SimpleNamespace
import src.graph.graph_builder as gb

class DummyGraph:
    def add_graph_documents(self, graph_documents):
        return None

original_get_graph = gb.get_graph
original_transformer = gb.transformer

try:
    gb.get_graph = lambda: DummyGraph()
    gb.transformer = SimpleNamespace(
        convert_to_graph_documents=lambda chunks: [
            SimpleNamespace(nodes=[], relationships=[])
        ]
    )
    failed = gb.build_graph_from_chunks(["chunk"], batch_size=999)
finally:
    gb.get_graph = original_get_graph
    gb.transformer = original_transformer

results = TestResults()
results.check("zero-node extract flagged", len(failed) == 1)
results.check("failed index is 0", failed[0][0] == 0)
results.summary()


## 3. Community detection — unit test (offline)

Production uses Neo4j → NetworkX → Leiden → write back. Here we test **Leiden partitioning** on a tiny synthetic graph (no database).


In [ ]:
import networkx as nx
import igraph as ig
import leidenalg

G = nx.Graph()
G.add_edges_from([("a", "b"), ("b", "c"), ("d", "e")])  # two components

node_list = list(G.nodes())
node_index = {node: i for i, node in enumerate(node_list)}
edges = [(node_index[u], node_index[v]) for u, v in G.edges()]
ig_graph = ig.Graph(edges=edges, n=len(node_list))
partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition)

results = TestResults()
results.check("at least 2 communities", len(partition) >= 2)
# a,b,c should share a community; d,e another
comm_of = {}
for cid, members in enumerate(partition):
    for idx in members:
        comm_of[node_list[idx]] = cid
results.check("connected pair same community", comm_of["a"] == comm_of["b"])
results.check("isolated component separate", comm_of["c"] != comm_of["d"] or comm_of["a"] != comm_of["e"])
results.summary()


## 4. Query routing (LOCAL vs GLOBAL)

Implementation: `src/retrieval/graph.py`

| Step | Function | Behavior |
|------|----------|----------|
| Route | `classify_query` | Keyword override for corpus-meta questions → **GLOBAL**; else LLM **LOCAL/GLOBAL** |
| Local | `extract_entities` + `local_search` | Cypher path expansion around entity `id`s |
| Global | `global_search` | Read `Community.summary`, map-reduce style LLM synthesis |
| Entry | `smart_query` | Chooses local vs global automatically |

API: `POST /query` → `query_graph` → `smart_query` only (graph path).


In [ ]:
from unittest.mock import patch
from src.retrieval import graph as graph_retrieval

results = TestResults()

# Keyword override should force GLOBAL without calling LLM
q_global = "What topics are in the knowledge base?"
kind = graph_retrieval.classify_query(q_global)
results.check("meta question -> GLOBAL", kind == "GLOBAL")

with patch.object(graph_retrieval.llm, "invoke", return_value=type("R", (), {"content": "LOCAL"})()):
    kind2 = graph_retrieval.classify_query("Who founded Apex Solutions?")
    results.check("specific entity question -> LOCAL", kind2 == "LOCAL")

with patch.object(graph_retrieval, "classify_query", return_value="GLOBAL"):
    with patch.object(graph_retrieval, "global_search", return_value="global-answer"):
        out = graph_retrieval.smart_query("Summarize everything")
        results.check("smart_query uses global path", out == "global-answer")

with patch.object(graph_retrieval, "classify_query", return_value="LOCAL"):
    with patch.object(graph_retrieval, "extract_entities", return_value=["Leadership"]):
        with patch.object(graph_retrieval, "local_search", return_value="local-answer"):
            out2 = graph_retrieval.smart_query("What is Transactional Leadership?")
            results.check("smart_query uses local path", out2 == "local-answer")

results.summary()


## 5. Search strategies & hybrid pipeline

| Strategy | Module | When |
|----------|--------|------|
| **Graph local/global** | `graph.smart_query` | `/query` |
| **Vector (basic RAG)** | `vector.vector_search` | Milvus top-k chunks |
| **Hybrid** | `hybrid.hybrid_search` | Parallel graph + vector |
| **Rerank** | `reranker.rerank_and_merge` | CrossEncoder merges candidates |
| **Hybrid answer** | `hybrid.hybrid_answer` | Reranked context → final LLM |
| **Agent** | `agent.run_agent` | Enhancer + deep agent; KB tool uses hybrid |

Query enhancer (`query_enhancer.enhance_query`) rewrites questions before hybrid/agent retrieval.


In [ ]:
from unittest.mock import patch
from src.retrieval.query_enhancer import enhance_query
from src.retrieval.reranker import rerank_and_merge

results = TestResults()
results.check("empty query unchanged", enhance_query("   ") == "")

hybrid_stub = {
    "graph_result": "Graph fact: Alice KNOWS Bob",
    "vector_results": [
        {"text": "Chunk about Alice", "origin": "vector", "score": 0.9},
        {"text": "Chunk about Bob", "origin": "vector", "score": 0.8},
    ],
}

with patch("src.retrieval.reranker.get_reranker") as mock_reranker:
    mock_reranker.return_value.predict.return_value = [0.2, 0.95, 0.5]
    ranked = rerank_and_merge("Who is Alice?", hybrid_stub, top_k=2)

results.check("rerank returns top_k", len(ranked) == 2)
results.check("has rerank_score", all("rerank_score" in c for c in ranked))
results.check("graph candidate included", any(c.get("origin") == "graph" for c in ranked))
results.check("sorted by rerank_score", ranked[0]["rerank_score"] >= ranked[1]["rerank_score"])

results.summary()


In [ ]:
# hybrid_answer end-to-end with mocks (no Milvus/Neo4j/LLM network)
from unittest.mock import patch
from src.retrieval import hybrid as hybrid_mod

results = TestResults()

fake_hybrid = {
    "graph_result": "Entity X --RELATED--> Y",
    "vector_results": [{"text": "supporting chunk", "origin": "vector", "score": 0.77}],
    "graph_error": None,
    "vector_error": None,
}

with patch.object(hybrid_mod, "hybrid_search", return_value=fake_hybrid):
    with patch.object(hybrid_mod, "rerank_and_merge", return_value=[{"origin": "graph", "text": "Entity X --RELATED--> Y", "rerank_score": 1.0}]):
        with patch.object(hybrid_mod.llm, "invoke", return_value=type("R", (), {"content": "Final grounded answer"})()):
            answer = hybrid_mod.hybrid_answer("Question?", retrieval_question="focused query")

results.check("hybrid returns LLM content", answer == "Final grounded answer")

with patch.object(hybrid_mod, "hybrid_search", return_value={"graph_result": None, "vector_results": []}):
    empty = hybrid_mod.hybrid_answer("anything")
    results.check("empty KB message", "couldn't find" in empty.lower())

results.summary()


## 6. Query service layers (API mapping)

```text
POST /query        → query_graph      → smart_query (graph router only)
POST /query/hybrid → query_hybrid     → enhance_query → hybrid_answer
POST /query/agent  → query_agent      → enhance_query → run_agent (KB + Tavily)
```

`gradio_app.py` uses **`/upload`** and **`/query/agent`**.


In [ ]:
from unittest.mock import patch
from src.services import query_service

results = TestResults()

with patch("src.services.query_service.smart_query", return_value="graph-only"):
    r = query_service.query_graph("test")
    results.check("query_graph delegates", r == "graph-only")

with patch("src.services.query_service.enhance_query", return_value="enhanced"):
    with patch("src.services.query_service.hybrid_answer", return_value="hybrid-ans") as ha:
        r2 = query_service.query_hybrid("hello")
        results.check("query_hybrid returns answer", r2 == "hybrid-ans")
        results.check("enhancer used", ha.call_args.kwargs.get("retrieval_question") == "enhanced")

with patch("src.services.query_service.enhance_query", return_value="enhanced"):
    with patch("src.services.query_service.run_agent", return_value={"answer": "agent-ans"}) as ra:
        r3 = query_service.query_agent("hello")
        results.check("query_agent dict answer", r3["answer"] == "agent-ans")
        results.check("agent gets retrieval_query", ra.call_args.kwargs.get("retrieval_query") == "enhanced")

results.summary()


## 7. Agent KB gate (relevance threshold)

`hybrid_knowledge_base` checks Milvus top-1 score via `kb_relevance_score`. Below **0.25**, the agent is told to use Tavily instead.


In [ ]:
from unittest.mock import patch
from src.agent import agent as agent_mod

results = TestResults()
results.check("threshold is 0.25", agent_mod.RELEVANCE_THRESHOLD == 0.25)

with patch.object(agent_mod, "kb_relevance_score", return_value=0.1):
    with patch.object(agent_mod, "hybrid_answer") as ha:
        msg = agent_mod.hybrid_knowledge_base.invoke({"question": "obscure topic"})
        results.check("low score blocks KB", "KNOWLEDGE_BASE_INSUFFICIENT" in msg)
        results.check("hybrid not called", ha.call_count == 0)

with patch.object(agent_mod, "kb_relevance_score", return_value=0.9):
    with patch.object(agent_mod, "hybrid_answer", return_value="KB hit"):
        msg2 = agent_mod.hybrid_knowledge_base.invoke({"question": "on-topic"})
        results.check("high score uses hybrid", msg2 == "KB hit")

results.summary()


## 8. Full pipeline checklist (for live demo)

Run with services up and `GRAPHRAG_RUN_INTEGRATION=1`:

1. Start API + worker + Redis + MinIO + Neo4j + Milvus
2. `curl -F file=@sample.pdf http://127.0.0.1:8002/api/v1/upload`
3. Wait until document status is `done` in metadata store
4. `POST /query` — graph routing only
5. `POST /query/hybrid` — graph + vector + rerank
6. `POST /query/agent` — planner todos + tools
7. Optional: `python gradio_app.py` for UI

The next cell runs **smoke integration** only when enabled.


In [ ]:
if not RUN_INTEGRATION:
    print("Skipping integration smoke tests. Set GRAPHRAG_RUN_INTEGRATION=1 to enable.")
else:
    from src.config.settings import settings
    results = TestResults()
    results.check("NEO4J_URI set", bool(settings.neo4j_uri))
    results.check("MILVUS_URI set", bool(settings.milvus_uri))
    results.check("LLM_API_KEY set", bool(settings.llm_api_key))

    try:
        from src.storage.neo4j_client import get_driver
        driver = get_driver()
        with driver.session() as session:
            count = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
        print(f"Neo4j node count: {count}")
        results.check("neo4j reachable", count >= 0)
    except Exception as exc:
        results.check("neo4j reachable", False, str(exc))

    try:
        from src.retrieval.vector import vector_search
        hits = vector_search("test connectivity", top_k=1)
        results.check("milvus search ok", isinstance(hits, list))
    except Exception as exc:
        results.check("milvus search ok", False, str(exc))

    results.summary()


## 9. Run all unit tests in one shot

Re-run this cell after changing code under `src/`.


In [ ]:
# Re-run core offline suite (aggregated)
from unittest.mock import patch
from types import SimpleNamespace
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.retrieval.reranker import rerank_and_merge

failures = []

def run(name, fn):
    try:
        fn()
        print(f"OK  {name}")
    except Exception as e:
        failures.append((name, e))
        print(f"ERR {name}: {e}")

def t_upload():
    allowed = {".pdf", ".docx", ".pptx", ".xlsx"}
    assert ".pdf" in allowed and ".txt" not in allowed
run("upload", t_upload)

def t_chunk():
    splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
    ch = splitter.split_documents([Document(page_content="word " * 200)])
    assert len(ch) >= 2
run("chunk", t_chunk)

# 3 graph fail
import src.graph.graph_builder as gb
def t_graph():
    class DG:
        def add_graph_documents(self, g): pass
    o1, o2 = gb.get_graph, gb.transformer
    try:
        gb.get_graph = lambda: DG()
        gb.transformer = SimpleNamespace(convert_to_graph_documents=lambda c: [SimpleNamespace(nodes=[], relationships=[])])
        assert len(gb.build_graph_from_chunks(["x"], batch_size=999)) == 1
    finally:
        gb.get_graph, gb.transformer = o1, o2
run("graph_empty", t_graph)

# 4 router
from src.retrieval import graph as gr
def t_router():
    assert gr.classify_query("what is in the knowledge base?") == "GLOBAL"
run("router", t_router)

# 5 rerank
def t_rerank():
    with patch("src.retrieval.reranker.get_reranker") as m:
        m.return_value.predict.return_value = [0.9, 0.1]
        r = rerank_and_merge("q", {"graph_result": "g", "vector_results": []}, top_k=1)
        assert len(r) == 1
run("rerank", t_rerank)

print("\n" + "=" * 60)
if failures:
    raise AssertionError(f"{len(failures)} aggregated failures: {failures}")
print("Aggregated offline suite: ALL PASSED")
